# Controlled signal discovery results

This notebook reads the controlled run built on the actual RelAmazon schema, standard full 39-candidate pool, and shared signal-scorer prompt construction. It compares ranking quality against constructed synthetic qrels and shows per-condition rankings for Random, MiniLM, Qwen3-Reranker, and Decider. The builder copies schema metadata only, not source records. This does **not** rerun the models.

The suite covers direct attributes, one- and two-hop paths, relationship-only signals, multiple relevant signals, distractors, AND/OR logic, and temporal cases. Its grades are author-defined synthetic qrels—not human-validated judgments—so treat these scores as a controlled capability check, not as evidence of performance on real user conditions. Model inputs now share the standard RelAmazon signal options and prompt template. Rank metrics are comparable across models; raw model scores are not, because each scorer has its own score scale.

In [ ]:
from pathlib import Path
import json
import os
MPL_CONFIG = Path('/tmp/simbench-matplotlib')
MPL_CONFIG.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RESULT_DIR = Path('/storage/lcostes/ray-share/experiments/simbench-signals/controlled-rel-amazon-v2-ray-20261012')
SUMMARY_PATH = RESULT_DIR / 'summary.json'
TASKS_PATH = RESULT_DIR / 'summary-tasks.jsonl'

if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(f'Missing aggregate results: {SUMMARY_PATH}')
if not TASKS_PATH.is_file():
    raise FileNotFoundError(f'Missing per-task results: {TASKS_PATH}')

summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
task_rows = [json.loads(line) for line in TASKS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
task_count = len({row['task_id'] for row in task_rows})
scorer_names = sorted({row['scorer'] for row in task_rows})
print(f"Benchmark: {summary['benchmark']} | tasks: {task_count} | scorers: {scorer_names} | rows: {len(task_rows)} | files: {RESULT_DIR}")
assert task_count == 25, f'Expected 25 tasks, found {task_count}'
assert all(row['candidate_count'] == 39 for row in task_rows), 'Expected the standard 39-candidate pool in every task'
assert set(scorer_names) == {'random', 'minilm', 'qwen3-reranker', 'decider'}

In [ ]:
# Aggregate comparison, macro-averaged equally over all 25 conditions.
aggregate = pd.DataFrame.from_dict(summary['scorers'], orient='index').rename_axis('scorer').reset_index()
metric_columns = ['ndcg@1', 'ndcg@3', 'ndcg@5', 'recall@1', 'recall@3', 'recall@5', 'mrr']
display(aggregate[['scorer', 'task_count'] + metric_columns].sort_values('ndcg@3', ascending=False).style.format({c: '{:.3f}' for c in metric_columns}))

In [ ]:
# Aggregate nDCG and recall curves. They measure complementary things:
# nDCG uses the grades and ranking order; recall@k asks whether all central signals were found.
plot_metrics = ['ndcg@1', 'ndcg@3', 'ndcg@5', 'recall@1', 'recall@3', 'recall@5']
ax = aggregate.set_index('scorer')[plot_metrics].T.plot(figsize=(11, 5), marker='o', linewidth=2)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Macro average across conditions')
ax.set_xlabel('Metric')
ax.set_title('Controlled signal-discovery ranking quality')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=25, ha='right')
plt.legend(title='Method', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Per-condition nDCG@3 heatmap. This exposes cases hidden by the macro average.
per_task = pd.DataFrame(task_rows)
task_order = list(dict.fromkeys(per_task['task_id']))
scorer_order = list(summary['scorers'])
heat = per_task.pivot(index='scorer', columns='task_id', values='ndcg@3').reindex(index=scorer_order, columns=task_order)
fig, ax = plt.subplots(figsize=(20, 7))
image = ax.imshow(heat.to_numpy(), vmin=0, vmax=1, cmap='viridis', aspect='auto')
short_labels = [task.replace('-', '\n') for task in task_order]
ax.set_xticks(range(len(task_order)), short_labels, fontsize=8)
ax.set_yticks(range(len(scorer_order)), scorer_order)
ax.set_title('nDCG@3 by method and synthetic condition')
for i in range(len(scorer_order)):
    for j in range(len(task_order)):
        ax.text(j, i, f'{heat.iloc[i, j]:.2f}', ha='center', va='center', color='white' if heat.iloc[i, j] < 0.55 else 'black')
fig.colorbar(image, ax=ax, label='nDCG@3')
plt.tight_layout()
plt.show()

In [ ]:
# Readable cross-method comparison for one condition. Change this ID to inspect another.
SELECT_TASK = 'product-customer-behavior'
selected = per_task[per_task['task_id'] == SELECT_TASK].copy()
if selected.empty:
    raise KeyError(f'No such task: {SELECT_TASK}; available: {task_order}')
print(f"{selected.iloc[0]['reasoning_type']} | {selected.iloc[0]['condition']}")

top_rows = []
for _, row in selected.iterrows():
    for signal in row['top_signals']:
        top_rows.append({
            'method': row['scorer'],
            'reasoning type': row['reasoning_type'],
            'rank': signal['rank'],
            'grade': signal['grade'],
            'signal': signal['description'],
            'nDCG@3': row['ndcg@3'],
        })
top_table = pd.DataFrame(top_rows)
display(top_table[top_table['rank'] <= 5].sort_values(['method', 'rank']).style.format({'nDCG@3': '{:.3f}'}))

In [ ]:
# Compact per-condition metric table. Useful for copying into notes.
condition_metrics = per_task[['scorer', 'reasoning_type', 'task_id', 'candidate_count', 'ndcg@3', 'recall@3', 'mrr']].sort_values(['reasoning_type', 'task_id', 'scorer'])
display(condition_metrics.style.format({'ndcg@3': '{:.3f}', 'recall@3': '{:.3f}', 'mrr': '{:.3f}'}))

## Reading these results

A higher nDCG means the relevant signals, including their relevance grades, appear earlier in the ranking. Recall@k measures how many of the grade-3 central signals appear in the first k results. MRR is the reciprocal rank of the first grade-3 signal.

There are 25 curated synthetic conditions, but some reasoning categories have only one task, so per-category conclusions are not statistically robust. Inspect the heatmap and top-signal table alongside the macro average. The qrels encode the suite author's intended mapping; they do not establish which signals people prefer for real RelAmazon conditions. These results use the real schema and same 39 signal options as standard runs, but compare only when the standard runs use the same scorer/prompt implementation.